# DA3: selecting columns

__We are practicing selecting columns from our data and then
creating new columns and then using them for plotting.__

In [ ]:
import pandas as pd
from plotnine import *
import numpy as np

surveys = pd.read_csv("data/surveys.csv")

surveys.columns

# Scale the 'weight' column manually creating a new column
We use the __.mean()__ and __.std()__ methods to calculate mean & standard deviation for scaling.

In [ ]:
surveys['weight_scaled'] = (surveys['weight'] - surveys['weight'].mean()) / surveys['weight'].std()

___NB: In geom_jitter the width parameter controls the horizontal jitter.
And in geom_hline we use yintercept to position the horizontal and the color and 
size parameters to set the style of the line.___

In [ ]:
p = (
    ggplot(surveys, aes(x='species_id', y='weight_scaled')) +
    geom_jitter(width=0.1) +
    geom_hline(yintercept=0, color="blue", size=1)
)

p.show()

__Let's say we define a category of 'big' animal as being greater than average weight and hindfoot length greater than median length and we create a new column that has an entry of 'big' if the animal meets these criteria.__
__This example uses numpys .select() method.__

In [ ]:
conditions = [
    (surveys['weight'] >= surveys['weight'].mean()) & 
    (surveys['hindfoot_length'] >= surveys['hindfoot_length'].median())]


choices =  ['big']
surveys["large_animal"] = np.select(conditions, choices, default="")

In [ ]:
# to display in a nice table all entries including new column
from IPython.display import display

# Display DataFrame with all columns
display(surveys)

## Add some chaining examples

In [ ]:
# Single operation
surveys[["record_id", "hindfoot_length"]].copy()

In [ ]:
# If we wanted to combine it with creating a new column we could do:
# select the required columns and store in a new data set
subset_surveys = surveys[["record_id", "hindfoot_length"]].copy()

# take the new data set and calculate the new column
subset_surveys["hindfoot_length_cm"] = subset_surveys["hindfoot_length"] / 10

So what is wrong about the above?





In [ ]:
# We are creating intermediate data structures for limited re-use (subset_surveys)
# we can instead chain the operation and use a transient (lambda function)
(
  surveys
  .loc[:, ['record_id', 'hindfoot_length']]
  .assign(hindfoot_length_cm = lambda df: df["hindfoot_length"] / 10)
)

In [ ]:
# This can be extended to more complex chains
(
  surveys
  .loc[:, ['record_id', 'weight']]          # select columns
  .rename(columns = {'weight': 'weight_g'}) # rename column
  .head()                                   # show first few rows
)

## Add some manipulating row examples (Chap10)
May need to run this in terminal:
mamba install conda-forge::missingno

In [ ]:
# Load missingno
import missingno as msno

In [ ]:
# Sorting rows by year
surveys.sort_values(by = "year")

In [ ]:
# if we want year descending and hf_length ascending order....
surveys.sort_values(
    by = ["year", "hindfoot_length"],
    ascending = [False, True]
)

In [ ]:
# Unique values in column (Note we need to drop any duplicates (There's a method for that...)
surveys[["sex"]].drop_duplicates()

In [ ]:
# Can extend this to partition on sex & year
surveys[["sex", "year"]].drop_duplicates()

# Filtering by condition - revisiting logicals

In [ ]:
# create a simple 1d structure for concept
some_years = pd.Series([1985, 1990, 1999, 1995, 2010, 2000])
result = some_years < 2000

result

In [ ]:
# extending idea to both conditions have to be true (logical AND)
result = (some_years > 1990) & (some_years < 2000)

result

In [ ]:
# With logical OR only one or the other of the conditions has to be true
result = (some_years < 1990) | (some_years > 2000)

result

In [ ]:
# Lets extend this to our example data to test one condition
# Recall that if we select by subsetting with logical vector we only return TRUE elements
surveys[surveys["hindfoot_length"] > 31]

In [ ]:
# There is also a (more readable) .query() method to do the same
surveys.query("hindfoot_length > 31")

## Were you missing Missing Data? 

In [ ]:
# We can just drop missing data using .dropna() method
# Why might this cause other problems?
surveys.dropna(subset = ["weight", "hindfoot_length"])

In [ ]:
# It can be combined with filters like this
surveys[
    (surveys["weight"].notna()) &
    (surveys["hindfoot_length"].notna()) &
    (surveys["hindfoot_length"] > 40)
]

## add grouped examples

In [ ]:
surveys = pd.read_csv("data/surveys.csv")
infections = pd.read_csv("data/infections.csv")


In [ ]:
surveys.groupby("sex")["weight"].agg(
    weight_mean = "mean",
    weight_sd = "std"
).reset_index()

In [ ]:
surveys.query("year == 1982").shape[0]